<a href="https://colab.research.google.com/github/antonidesjardins15/capstone-2026_beta/blob/main/CAPSTONE_D%C3%89CHET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data manipulation

## Create Delay Features

Engineer new features to capture the delay between scheduling and appointment. creation des fonctionnalie pour saisir le délai entre la prise de rendez-vous et le rendez-vous : lead_days (delay in days), 'lead_hours' (delay in hours), and 'is_same_day' (a boolean indicating if the appointment is on the same day it was scheduled)

In [ ]:
# Creation des nouvelles function
# *Creation de 'lead_hours' (resultat en timedelta)
df['lead_hours'] = (df['appointment_day'] - df['scheduled_day']).dt.total_seconds() / 3600 # '.dt.total_seconds()' : Convertit les resultats en seconde pour faire des calculs
                                                                                            # ...
# Creation de 'is_same_day'
df['is_same_day'] = (df['lead_days'] == 0) # creation dune variable booleenne (Vrai ou Faux).
                                            # A partir de 'lead_day', True = rendez-vous pris le jour-meme. Faux =  rendez-vous a été pris au moins un jour à l'avance
print("Delay features 'lead_hours' and 'is_same_day' created:")
df.head()

## Extract Calendar Features

Extraire les caractéristiques (Datetime) liées aux colonnes 'scheduled_day' et 'appointment_day' en utilisant '.dt' de pandas pour creer de nouvelles colonnes de caractéristiques temporelles (jour de la semaine, le mois et l'heure prévue).

Cette etape pour décomposer une date complète en informations numériques afin de pouvoir faire des calculs et anlayser les probalilte qu'un patient se presente ou pas.

In [ ]:
# '.dt.dayofweek' pour extraction du jour de la semaine
# Extrait le jour de la semaine chiffre entier (ex. lundi = 0, mardi = 1). Permer de determinr les tendences du no-show selon la journee de prise de rendez-vous
df['scheduled_day_of_week'] = df['scheduled_day'].dt.dayofweek
df['appointment_day_of_week'] = df['appointment_day'].dt.dayofweek

# 'dt.month' pour extraction du mois
# extrait le numero du mois (1 a 12). Permet de detecter des tendences saisonnier
df['scheduled_month'] = df['scheduled_day'].dt.month
df['appointment_month'] = df['appointment_day'].dt.month

# 'dt.hour' pour extraction de l'heure
# Extrait l'heure de la journee de la prise de rendez-vous (0 a 23). Permet de detecter des tendences selon moment de la journee de la prise de rendez-vous
df['scheduled_hour'] = df['scheduled_day'].dt.hour

print("Calendar features 'scheduled_day_of_week', 'appointment_day_of_week', 'scheduled_month', 'appointment_month', and 'scheduled_hour' created:")
df.head()

## *Engineer Patient History Features

Cette partie a pour but d'utiliser le comportement passé d'un patient pour prédire son comportement futur. Cette etape estconfiermer la tendence corpetementale, c'est 'a dire que si un patient a régulièrement manqué ses rendez-vous dans le passé, il est statistiquement plus probable qu'il manque le prochain.

Cette etape s'agit de developper des fonctionnalités basées sur l'historique de chaque patient : num_prev_appointments » (le nombre de rendez-vous précédents pour chaque patient) et 'prev_no_show_rate' (le taux d'absentéisme pour leurs rendez-vous précédents).

In [ ]:
# triage des donnees par 'patient_id' pour aller chercher rendez-vous du même patient ensemble
# triage des donnees par 'scheduled_day' ranger les rendez-vous du plus ancien au plus récent
df = df.sort_values(by=['patient_id', 'scheduled_day']).reset_index(drop=True) # 'reset_index(drop=True)' : garder l'ordre dans les lignes apres le triage (rangé par personne et par temps)

df['num_prev_appointments'] = df.groupby('patient_id').cumcount() # 'groupby('patient_id')' : isole 'patient_id' dans un groupe. cumcount(): compte les lignes par 'patient_id' une par une, en commençant par 0
df['prev_no_show_rate'] = df.groupby('patient_id')['no_show'].expanding().mean().shift(1).reset_index(level=0, drop=True) # 'expanding().mean()' : Calcule la moyenne qui s'eleve au fur a mesure que ca compte le no-show par 'patient_id'. shift(1): recommande pour eviter une fuite de données et des performances artificiellement excellentes lors du test
df['prev_no_show_rate'] = df['prev_no_show_rate'].fillna(0) # puisque 'shift(1)' a créé un vide (NaN) a la premiere ligne de 'prev_no_show_rate', on utilise 'fillna(0)' pour ajouter un 0 la premiere ligne de

print("Patient history features 'n_prev_appointments' and 'prev_no_show_rate' created")
df.head()

## Patient Geographic Features

Cette etape a pour but de capturer l'Influence Géographique pour determiner si le quartier où vit un patient peut être un facteur influancant la probabilite qu'un patient se présente ou pas à son rendez-vous. Selon le qaurtier peut nous donner un indice socio-économiques dans lequel influence le risque de no-show (ex. acces au transport)

En calculant ensuite le taux moyen de no-show par quartier, nous agrégeons des informations sur le comportement de la population locale.

In [ ]:
neigh_no_show_rate = df.groupby('neighborhood')['no_show'].mean()
df['neigh_no_show_rate'] = df['neighborhood'].map(neigh_no_show_rate)

print("Neighborhood no-show rate ('neigh_no_show_rate') created:")
df.head()

## Patient profile features - 60 years old or more, genre, age_bucket, num_chronic_disease

Creation des colonnes avec les caracteristique. Ca permettra à simplifier les informations sur le profil des patients et faciliter la comparaison statistique.

Le but est de voir se ces caracteristiques peut aussi avoir une influence au no-show

In [ ]:
# genre
df['is_male'] = np.where(df['gender'] == 'M', 1, 0) # 1 = Homme, 0 = Femme

# Old or not
df['is_old'] = np.where(df['age'] >= 60, 1, 0) # Si le patient a 60 ans ou plus = 1, sinon = 0

# Groupement par tranches d'âge (age_bucket)
bins = [0, 17, 30, 50, 64, df['age'].max()] # 'bins' pour creation de groupe (0 a 17, 18 a 30, etc.). .max() pour l'âge le plus élevé
labels = ['0-17', '18-30', '31-50', '51-64', '65+'] # 'labels' pour creer un nom a ces groupes
df['age_bucket'] = pd.cut(df['age'], bins=bins, labels=labels, right=True, include_lowest=True)

# Determination de la sante globale par patient (nombre de chronic desease par patient)
df['chronic_count'] = df['hypertension'] + df['diabetes'] + df['alcoholism'] + df['handicap'] # Ex. Si un patien a 'Hypertension' + 'Diabetes' = 2

print("Patient profile features 'age_bucket' and 'chronic_count' created. Displaying the first 5 rows:")
df.head()

print("Patient profile features 'is_male' and 'is_old' created:")
df.head()

## *3) Numpy Recap
Cette partie est pour aller chercher les features de notre cible de prédiction "no-show"

## Analyse descriptive d'une colonne numérique

In [ ]:
target_col = 'no_show'


# Sélectionne toutes les colonnes numériques, en excluant la colonne cible
numeric_cols = df.drop(columns=[target_col]).select_dtypes(include=[np.number]).columns.tolist()

if not numeric_cols:
    raise ValueError("No numeric columns found besides the target.")

# * Option d'une colonne
col_demo = 'is_old'

# Vérifie si la colonne choisie est bien numérique et existe
if col_demo not in numeric_cols:
    raise ValueError(f"Column '{col_demo}' is not a numeric column or does not exist.")

x = df[col_demo].to_numpy()

print("Demo column:", col_demo)
print("Type:", type(x))
print("Shape:", x.shape)
print("Mean:", x.mean())
print("Std:", x.std())

mask = x > np.median(x)
print("Share above median:", mask.mean())

## 4) Data visualization

## *Analyse Visuelle - Caractéristiques Numériques vs. Cible

Créer des boîtes à moustaches (box plots) pour analyser la distribution des principales caractéristiques numériques (age, lead_days, lead_hours, num_prev_appointments, prev_no_show_rate, neigh_no_show_rate, chronic_count) en fonction du statut de non-présentation (no_show: 0 pour 'présent', 1 pour 'non-présent').

creation of box plots for the specified numerical features against the 'no_show' target variable, following the detailed instructions provided.

Apres essaie, je voulais seulement montrer les numériques categorielles les plus pertientes pour le model de prediction de no-show

In [ ]:
numerical_features_for_boxplot = ['age', 'lead_days', 'lead_hours', 'num_prev_appointments', 'prev_no_show_rate', 'neigh_no_show_rate', 'chronic_count']

num_features = len(numerical_features_for_boxplot)
num_cols = 3 # Number of columns for subplots
num_rows = (num_features + num_cols - 1) // num_cols # Calculate number of rows needed

plt.figure(figsize=(num_cols * 5, num_rows * 4)) # Adjust figure size dynamically

for i, feature in enumerate(numerical_features_for_boxplot):
    plt.subplot(num_rows, num_cols, i + 1) # Create a subplot for each feature
    sns.boxplot(x='no_show', y=feature, data=df)
    plt.title(f'Distribution of {feature} by No-Show Status')
    plt.xlabel('No-Show (0=Present, 1=No-Show)')
    plt.ylabel(feature)

plt.tight_layout()
plt.show()

## *Analyse Visuelle - Caractéristiques Catégorielles/Binaires vs. Taux de Non-Présentation

Examiner la relation entre les caractéristiques catégorielles et binaires (`gender`, `is_male`, `is_old`, `scholarship`, `hypertension`, `diabetes`, `alcoholism`, `handicap`, `sms_received`, `scheduled_day_of_week`, `appointment_day_of_week`, `scheduled_month`, `appointment_month`, `scheduled_hour`, `age_bucket`) et le taux de non-présentation (`no_show`).

creation of bar plots for the specified categorical and binary features, showing their relationship with the no_show rate, as described in the subtask

In [ ]:
categorical_features_for_barplot = ['gender', 'is_male', 'is_old', 'scholarship', 'hypertension', 'diabetes', 'alcoholism', 'handicap', 'sms_received', 'scheduled_day_of_week', 'appointment_day_of_week', 'scheduled_month', 'appointment_month', 'scheduled_hour', 'age_bucket']

num_features_cat = len(categorical_features_for_barplot)
num_cols_cat = 3 # Number of columns for subplots
num_rows_cat = (num_features_cat + num_cols_cat - 1) // num_cols_cat # Calculate number of rows needed

plt.figure(figsize=(num_cols_cat * 6, num_rows_cat * 5)) # Adjust figure size dynamically

for i, feature in enumerate(categorical_features_for_barplot):
    plt.subplot(num_rows_cat, num_cols_cat, i + 1) # Create a subplot for each feature
    sns.barplot(x=feature, y='no_show', data=df, errorbar=None) # errorbar=None to remove error bars
    plt.title(f'No-Show Rate by {feature}')
    plt.xlabel(feature)
    plt.ylabel('No-Show Rate')
    plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for better readability

plt.tight_layout()
plt.show()

## *Analyse Visuelle - Caractéristiques Catégorielles/Binaires vs. Taux de Non-Présentation

Calculer et visualiser une matrice de corrélation (heatmap) pour toutes les caractéristiques numériques pertinentes et la variable cible no_show. Cela permettra de comprendre les relations linéaires entre les variables et d'identifier la multicolinéarité.

calculation of the correlation matrix for all relevant numerical features, including the target variable, and then visualize it using a Seaborn heatmap, as outlined in the subtask.

In [ ]:
relevant_numerical_features = [
    'age', 'scholarship', 'hypertension', 'diabetes', 'alcoholism', 'handicap',
    'sms_received', 'lead_days', 'lead_hours', 'scheduled_day_of_week',
    'appointment_day_of_week', 'scheduled_month', 'appointment_month', 'scheduled_hour',
    'num_prev_appointments', 'prev_no_show_rate', 'neigh_no_show_rate', 'is_male', 'is_old', 'chronic_count',
    'no_show' # Target variable
]

# Calculate the correlation matrix
correlation_matrix = df[relevant_numerical_features].corr()

# Plotting the heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Numerical Features and No-Show Status')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Analyse de Corrélation - Caractéristiques Numériques et Cible V.2

Pour simplifier l'analyse, nous allons extraire et afficher les coefficients de corrélation de toutes les caractéristiques pertinentes spécifiquement avec la variable cible `no_show`. Cela permet d'identifier rapidement les facteurs les plus influents.



In [ ]:
# Assurez-vous que 'no_show' est dans la liste des caractéristiques numériques pertinentes
# et qu'elle est de type numérique (ce qui a été fait lors du nettoyage).
relevant_numerical_features_for_target_corr = [
    'age', 'scholarship', 'hypertension', 'diabetes', 'alcoholism', 'handicap',
    'sms_received', 'lead_days', 'lead_hours', 'scheduled_day_of_week',
    'appointment_day_of_week', 'scheduled_month', 'appointment_month', 'scheduled_hour',
    'neigh_no_show_rate', 'is_male', 'is_old', 'chronic_count',
    'no_show' # Variable cible
]

# Calculer la matrice de corrélation complète (si elle n'est pas déjà calculée et à jour)
correlation_matrix = df[relevant_numerical_features_for_target_corr].corr()

# Extraire les corrélations avec 'no_show'
no_show_correlations = correlation_matrix['no_show'].sort_values(ascending=False)

# Exclure la corrélation de 'no_show' avec elle-même pour une meilleure lisibilité
no_show_correlations = no_show_correlations.drop('no_show')

print("Corrélation de chaque caractéristique avec 'no_show' (triée):")
display(no_show_correlations)

# Vous pouvez aussi visualiser les corrélations principales sous forme de barres
plt.figure(figsize=(10, 8))
sns.barplot(x=no_show_correlations.values, y=no_show_correlations.index, palette='viridis')
plt.title('Corrélation des Caractéristiques avec le Taux de Non-Présentation')
plt.xlabel('Coefficient de Corrélation')
plt.ylabel('Caractéristique')
plt.tight_layout()
plt.show()



---



# 5) Machine Learning



---



Entraîner un RandomForestClassifier et utiliser l'importance par permutation pour classer quantitativement les caractéristiques en fonction de leur impact sur la prédiction des non-présentations. Ensuite, je résumerai toutes les découvertes, en mettant en évidence les caractéristiques les plus importantes pour la prédiction des non-présentations afin de guider le développement futur du modèle.

Code to prepare the data, preprocess it using a pipeline, train a RandomForestClassifier, calculate permutation importance, and then visualize and summarize the results, as detailed in the subtask.

In [ ]:
X = df.drop('no_show', axis=1)
y = df['no_show']

# Drop identifier and original date/time columns, and features replaced by engineered ones
features_to_drop = [
    'patient_id', 'appointment_id', 'scheduled_day', 'appointment_day',
    'gender', 'neighborhood', 'is_same_day'
]
X = X.drop(columns=features_to_drop)

# Identify categorical and numerical features for the preprocessor
categorical_features = ['age_bucket']
numerical_features = X.select_dtypes(include=np.number).columns.tolist()



# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # Keep numerical features as they are
)

# Create a pipeline with preprocessing and RandomForestClassifier
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
])




# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train the model
model_pipeline.fit(X_train, y_train)

# Get feature names after one-hot encoding
ohe_feature_names = model_pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_features)
# The 'remainder' passthrough columns are the numerical_features
all_feature_names = np.concatenate([ohe_feature_names, numerical_features])




# Calculate permutation importance
result = permutation_importance(model_pipeline, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

# Create a DataFrame for importance results
feature_importances = pd.DataFrame({
    'feature': all_feature_names,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std})

# Sort by importance and display
feature_importances = feature_importances.sort_values(by='importance_mean', ascending=False)
print("\nFeature importances based on permutation importance:")
print(feature_importances.head(15)) # Display top 15 for brevity

# Plotting the feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x='importance_mean', y='feature', data=feature_importances.head(20))
plt.title('Top 20 Feature Importances (Permutation Importance)')
plt.xlabel('Mean Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

# Evaluate model performance
y_pred = model_pipeline.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, model_pipeline.predict_proba(X_test)[:, 1])
print(f"\nROC AUC Score: {roc_auc:.4f}")

ConfusionMatrixDisplay.from_estimator(model_pipeline, X_test, y_test, cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()